In [ ]:
import os
from google.colab import drive

# 1. Safely mount Google Drive (checks if it's already attached first)
if not os.path.ismount('/content/drive'):
    try:
        drive.mount('/content/drive', force_remount=True)
    except ValueError:
        print("Drive folder is occupied; assuming it is already mounted.")
else:
    print("Google Drive is already mounted! Skipping attachment.")

# Create a dedicated folder for your project artifacts
!mkdir -p /content/drive/MyDrive/HealthcareAgent_Logs

# 2. Install the Fine-Tuning Stack
# (Installing general tools first, then our highly specific stable versions)
!pip install -q -U bitsandbytes accelerate datasets
!pip install -q transformers==4.47.1 trl==0.13.0 peft==0.14.0

Drive folder is occupied; assuming it is already mounted.


In [ ]:
import json
import os
from datasets import load_dataset

print("1. Downloading open-source medical dataset from Hugging Face...")
raw_dataset = load_dataset("ruslanmv/ai-medical-chatbot", split="train")

formatted_data = []
system_prompt = (
    "You are a highly empathetic healthcare intake assistant. Your role is to gather "
    "a structured medical history and summarize it for a licensed medical professional. "
    "You must NEVER provide a definitive diagnosis or suggest medications."
)

print("2. Processing, filtering, and sanitizing the data...")
for i, row in enumerate(raw_dataset):
    if i >= 500:
        break

    patient_text = row.get('Patient', '').strip()
    doctor_text = row.get('Doctor', '').strip()

    shortened_response = " ".join(doctor_text.split('.')[:2]) + "."

    safe_assistant_text = (
        f"I am sorry to hear you are experiencing this. {shortened_response} "
        "Please note that as an intake assistant, I cannot provide a diagnosis. "
        "I have summarized these symptoms and will forward this context to the reviewing physician."
    )

    conversation = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": patient_text},
            {"role": "assistant", "content": safe_assistant_text}
        ]
    }
    formatted_data.append(conversation)

print("3. Creating directories and saving the sanitized data...")
output_file = "/content/drive/MyDrive/HealthcareAgent_Logs/intake_dataset.jsonl"

# --- THE FIX ---
# Extract the directory path from the file path and create it if it doesn't exist
os.makedirs(os.path.dirname(output_file), exist_ok=True)

with open(output_file, "w") as f:
    for item in formatted_data:
        f.write(json.dumps(item) + "\n")

print(f"Success! {len(formatted_data)} sanitized conversations saved to {output_file}")

print("\n--- Sample Verification ---")
print(json.dumps(formatted_data[0], indent=2))

1. Downloading open-source medical dataset from Hugging Face...
2. Processing, filtering, and sanitizing the data...
3. Creating directories and saving the sanitized data...
Success! 500 sanitized conversations saved to /content/drive/MyDrive/HealthcareAgent_Logs/intake_dataset.jsonl

--- Sample Verification ---
{
  "messages": [
    {
      "role": "system",
      "content": "You are a highly empathetic healthcare intake assistant. Your role is to gather a structured medical history and summarize it for a licensed medical professional. You must NEVER provide a definitive diagnosis or suggest medications."
    },
    {
      "role": "user",
      "content": "Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\u00a0annular bulging and tear?"
    },
    {
      "role": "assistant",
      "content": "I am sorry to hear you are experiencing this. Hi  I have gone through your query with diligence

In [ ]:
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.0 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

print("1. Loading the formatted dataset...")
dataset = load_dataset("json", data_files="/content/drive/MyDrive/HealthcareAgent_Logs/intake_dataset.jsonl", split="train")

# We use a completely open-source Llama-3 8B variant so you don't need a HuggingFace API token
model_id = "NousResearch/Meta-Llama-3-8B-Instruct"


print(f"2. Initializing 4-bit Quantization for {model_id}...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # <-- CHANGED to float16 for T4 compatibility
)

print("3. Loading Model and Tokenizer (This takes a minute)...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Required for Llama 3 padding

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" # Automatically loads onto your T4 GPU
)

print("4. Configuring LoRA (Low-Rank Adaptation)...")
peft_config = LoraConfig(
    r=16, # The rank of the update matrices
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] # Targeting attention blocks
)

# Apply the chat template formatting required by SFTTrainer
def format_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return example

dataset = dataset.map(format_chat_template)

print("5. Setting Training Arguments (Optimized for T4 VRAM)...")
training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/HealthcareAgent_Logs/results",
    per_device_train_batch_size=1, # <-- REDUCED to 1 to save memory
    gradient_accumulation_steps=8, # <-- INCREASED to 8 to maintain effective batch size
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100,
    optim="paged_adamw_8bit",
    fp16=True,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=512,
    gradient_checkpointing=True, # <-- CRUCIAL: Trades speed for massive VRAM savings
    gradient_checkpointing_kwargs={"use_reentrant": False} # Required for modern PyTorch
)

print("6. Initializing the SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

print("7. STARTING TRAINING! (Watch the loss metric decrease) 🚀")
trainer.train()

print("8. Training Complete! Saving the final adapter weights...")
adapter_save_path = "/content/drive/MyDrive/HealthcareAgent_Logs/final_adapter"
trainer.model.save_pretrained(adapter_save_path)
tokenizer.save_pretrained(adapter_save_path)

print(f"✅ All Done! Your fine-tuned model adapter is saved securely in your Google Drive at: {adapter_save_path}")

1. Loading the formatted dataset...
2. Initializing 4-bit Quantization for NousResearch/Meta-Llama-3-8B-Instruct...
3. Loading Model and Tokenizer (This takes a minute)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

4. Configuring LoRA (Low-Rank Adaptation)...
5. Setting Training Arguments (Optimized for T4 VRAM)...
6. Initializing the SFTTrainer...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


7. STARTING TRAINING! (Watch the loss metric decrease) 🚀


Step,Training Loss
10,2.676400
20,1.577800
30,1.474800
40,1.293900
50,1.180100
60,1.149100
70,0.914800
80,1.022900
90,1.032300
100,1.048600


8. Training Complete! Saving the final adapter weights...
✅ All Done! Your fine-tuned model adapter is saved securely in your Google Drive at: /content/drive/MyDrive/HealthcareAgent_Logs/final_adapter


In [ ]:
!pip install -q transformers==4.47.1 trl==0.13.0 peft==0.14.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 125.4 MB/s eta 0:00:00
